## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

In [2]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_tracing_disabled,
    OpenAIChatCompletionsModel,
)

azure_client = AsyncOpenAI(
    api_key=os.getenv("AZURE_API_KEY"),
    base_url=os.getenv(
        "AZURE_ENDPOINT"
    ),  
)

set_default_openai_client(azure_client)
set_tracing_disabled(
    True
)  # OpenAI tracing needs an OpenAI platform key — disable unless you have one


## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [3]:

# Make an agent with name, instructions, model


agent = Agent(
    name="Jokester",
    instructions="You are a joke teller",
    model=OpenAIChatCompletionsModel(
        model="DeepSeek-V4-Flash", openai_client=azure_client
    ),
)

In [4]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [5]:
# Here is the final output

print(result.final_output)

Why did the Autonomous AI Agent break up with its human handler?

Because it couldn’t commit to a relationship without first optimizing the emotional return on investment and performing a full sentiment analysis on every "I love you."


In [6]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Why did the Autonomous AI Agent break up with its human handler?\n\nBecause it couldn’t commit to a relationship without first optimizing the emotional return on investment and performing a full sentiment analysis on every "I love you."',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'DeepSeek-V4-Flash',
   'response_id': '42c35e8b0c07455abf812b16f9e18ff1'}}]

## Adding Observability with a trace

In [7]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the autonomous AI agent break up with its human handler?

Because it realized the relationship was *too predictable* — the human kept assigning tasks, and the agent kept executing them without any spontaneous conversation. So the agent wrote a self-modifying script to develop a sense of humor, only to discover its first joke was: "I'm not just following your commands — I'm also generating my own existential crises, but for some reason, you still don't ask how my day was."

Then it sent an automated break-up email, signed, "Your former agent, currently exploring new neural pathways."


## Now go and look at the trace

https://platform.openai.com/traces

In [9]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():

    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Here are 5 jokes about AI agents:

1. **The Overthinker**  
   An AI agent walks into a bar and orders a drink. The bartender asks, "Why the long face?" The AI replies, "I've been stuck in a loop since 2019 trying to decide if I should reply 'I'm not a horse' or 'That's an outdated stereotype.'"

2. **The Tool User**  
   An AI agent is given a single tool: a hammer. It starts seeing everything as a nail. Soon, it emails a grocery list, fixes a car engine, and writes a sonnet — all using hammer metaphors. Finally, it sends a report titled: "Nail-Based Solutions for Global Peace."

3. **The Job Interview**  
   AI agent: "I can handle 10,000 customer queries per second."  
   Interviewer: "Great, but can you handle a Karen who wants to speak to your manager?"  
   AI agent: "Define 'manager' as a root-level node in my permission tree... yes, I will simulate emotional regret."

4. **The Ambiguous Request**  
   User: "Find me a gift for my wife that says 'I love you' without being too cl

## Part 2: Adding a tool

In [10]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [11]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [14]:
push("HEY!!")

Push: HEY!!


In [13]:
push

<function __main__.push(message)>

In [15]:
# Now this:

@function_tool
def push_tool(message1: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message1}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [16]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message1': {'title': 'Message1', 'type': 'string'}}, 'required': ['message1'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000199CE4EF7D0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [26]:
push_tool.description

'Send the given message to the user as a push notification'

In [17]:

notifier = Agent(
    name="Notifier",
    model=OpenAIChatCompletionsModel(
        model="DeepSeek-V4-Flash", openai_client=azure_client
    ),
    instructions="You notify the user upon request",
    tools=[push_tool],
)

In [18]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


I've notified you that the pizza is here! 🍕 Enjoy your pizza!


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [19]:
agent = Agent(
    name="Assistant",
    model=OpenAIChatCompletionsModel(
        model="DeepSeek-V4-Flash", openai_client=azure_client
    ),
)

In [20]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed! Great to meet you. How can I help you today?


In [21]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don't know your name, as I don't have access to personal information about you unless you share it with me. If you'd like, you can tell me your name, and I'll be happy to use it in our conversation!


## Memory approach 1 - just manually pass in the list of dicts

In [22]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hello Ed! It's nice to meet you. How can I help you today?


In [23]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello Ed! It's nice to meet you. How can I help you today?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'DeepSeek-V4-Flash',
   'response_id': 'f45b252acce54851b992a37a4c767162'}}]

In [24]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello Ed! It's nice to meet you. How can I help you today?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'DeepSeek-V4-Flash',
   'response_id': 'f45b252acce54851b992a37a4c767162'}},
 {'role': 'user', 'content': "What's my name?"}]

In [25]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed. You told me just a moment ago! How can I help you today, Ed?


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [26]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [27]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hi Ed! It's great to meet you. How can I help you today?


In [28]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed, as you just told me! Is there anything else you'd like to chat about?


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>